# 第13章 - BERT预训练

本notebook整合了以下内容:
- BERT模型架构详解
- 两个预训练任务:掩码语言模型(MLM)和下一句预测(NSP)
- WikiText-2数据集准备
- BERT预训练实现
- 使用预训练BERT表示文本

## 1. BERT模型概述

### 1.1 词表示的演进

**上下文无关表示**:
- Word2Vec, GloVe
- 同一个词总是相同的向量
- 问题:"bank"(银行/河岸)无法区分

**上下文相关表示**:
- **TagLM, CoVe, ELMo**: BiLSTM编码
  - 优势:双向上下文
  - 劣势:需要任务特定架构
  
- **GPT**: Transformer decoder
  - 优势:任务无关
  - 劣势:只有从左到右的单向上下文

- **BERT**: 结合两者优势!
  - ✓ 双向上下文(Transformer encoder)
  - ✓ 任务无关
  - ✓ 最小化架构改动
  - ✓ 端到端微调所有参数

In [ ]:
import torch
from torch import nn
from d2l import torch as d2l
import random

# 设置随机种子
torch.manual_seed(42)
random.seed(42)

### 1.2 BERT输入表示

**设计要求**:
1. 支持单个文本
2. 支持文本对(问答、自然语言推理等)
3. 区分不同序列

**Token结构**:
- 单文本: `<cls>` + tokens + `<sep>`
- 文本对: `<cls>` + tokens_a + `<sep>` + tokens_b + `<sep>`

**三种嵌入**:
1. **Token嵌入**: 词汇表查找
2. **Segment嵌入**: 区分序列A和B
   - 序列A: $\mathbf{e}_A$
   - 序列B: $\mathbf{e}_B$
3. **Position嵌入**: 位置信息(可学习的!)
   - 与Transformer的固定正弦编码不同
   - BERT使用可学习的位置嵌入

**最终输入**: Token嵌入 + Segment嵌入 + Position嵌入

In [ ]:
def get_tokens_and_segments(tokens_a, tokens_b=None):
    """获取BERT的输入序列的tokens和segments"""
    tokens = ['<cls>'] + tokens_a + ['<sep>']
    # 0表示段A, 1表示段B
    segments = [0] * (len(tokens_a) + 2)
    if tokens_b is not None:
        tokens += tokens_b + ['<sep>']
        segments += [1] * (len(tokens_b) + 1)
    return tokens, segments

# 测试
tokens, segments = get_tokens_and_segments(
    ['this', 'movie', 'is', 'great'], 
    ['i', 'like', 'it'])
print(f'Tokens: {tokens}')
print(f'Segments: {segments}')

### 1.3 BERTEncoder架构

**核心组件**:
- 基于Transformer Encoder
- 多层EncoderBlock堆叠
- 每个block包含:
  - 多头自注意力机制
  - 位置前馈网络
  - 层归一化
  - Dropout正则化

**BERT配置**:
- BERT-Base: 12层, 768隐藏单元, 12注意力头, 110M参数
- BERT-Large: 24层, 1024隐藏单元, 16注意力头, 340M参数

In [ ]:
# BERTEncoder实现
class BERTEncoder(nn.Module):
    """BERT编码器"""
    def __init__(self, vocab_size, num_hiddens, norm_shape, ffn_num_input,
                 ffn_num_hiddens, num_heads, num_layers, dropout,
                 max_len=1000, key_size=768, query_size=768, value_size=768,
                 **kwargs):
        super(BERTEncoder, self).__init__(**kwargs)
        # Token嵌入
        self.token_embedding = nn.Embedding(vocab_size, num_hiddens)
        # Segment嵌入(只有2个段)
        self.segment_embedding = nn.Embedding(2, num_hiddens)
        # 位置嵌入(可学习)
        self.pos_embedding = nn.Parameter(
            torch.randn(1, max_len, num_hiddens))
        # Transformer编码器块
        self.blks = nn.Sequential()
        for i in range(num_layers):
            self.blks.add_module(f"{i}", d2l.EncoderBlock(
                key_size, query_size, value_size, num_hiddens, norm_shape,
                ffn_num_input, ffn_num_hiddens, num_heads, dropout, True))

    def forward(self, tokens, segments, valid_lens):
        # 在以下代码段中，X的形状保持不变：(batch_size, max_len, num_hiddens)
        X = self.token_embedding(tokens) + self.segment_embedding(segments)
        X = X + self.pos_embedding.data[:, :X.shape[1], :]
        for blk in self.blks:
            X = blk(X, valid_lens)
        return X

# 测试BERTEncoder
vocab_size, num_hiddens, ffn_num_hiddens, num_heads = 10000, 768, 1024, 4
norm_shape, ffn_num_input, num_layers, dropout = [768], 768, 2, 0.2
encoder = BERTEncoder(vocab_size, num_hiddens, norm_shape, ffn_num_input,
                     ffn_num_hiddens, num_heads, num_layers, dropout)

# 创建示例输入
tokens = torch.randint(0, vocab_size, (2, 8))
segments = torch.tensor([[0, 0, 0, 0, 1, 1, 1, 1], [0, 0, 0, 1, 1, 1, 1, 1]])
valid_lens = torch.tensor([8, 7])
encoded_X = encoder(tokens, segments, valid_lens)
print(f'Encoder输出形状: {encoded_X.shape}')  # (batch_size, seq_len, num_hiddens)

## 2. 预训练任务

### 2.1 掩码语言模型(MLM)

**动机**:
- 传统语言模型是单向的(从左到右或从右到左)
- BERT需要双向上下文
- 但直接预测会导致"信息泄露"(模型能看到答案)

**解决方案**: 掩码语言模型
1. 随机选择15%的token进行预测
2. 对于每个选中的token:
   - 80%概率: 替换为`<mask>`
   - 10%概率: 替换为随机token
   - 10%概率: 保持不变

**为什么这样设计?**
- 80% `<mask>`: 主要训练目标
- 10% 随机: 增加鲁棒性,防止过拟合
- 10% 不变: 减少预训练-微调的差异(微调时没有`<mask>`)

In [ ]:
# MaskLM实现
class MaskLM(nn.Module):
    """BERT的掩码语言模型任务"""
    def __init__(self, vocab_size, num_hiddens, num_inputs=768, **kwargs):
        super(MaskLM, self).__init__(**kwargs)
        # MLP: 隐藏层 -> ReLU -> LayerNorm -> 输出层
        self.mlp = nn.Sequential(
            nn.Linear(num_inputs, num_hiddens),
            nn.ReLU(),
            nn.LayerNorm(num_hiddens),
            nn.Linear(num_hiddens, vocab_size)
        )

    def forward(self, X, pred_positions):
        """
        参数:
            X: BERTEncoder的输出 (batch_size, seq_len, num_hiddens)
            pred_positions: 需要预测的位置 (batch_size, num_pred_positions)
        返回:
            mlm_Y_hat: 预测的logits (batch_size, num_pred_positions, vocab_size)
        """
        num_pred_positions = pred_positions.shape[1]
        pred_positions = pred_positions.reshape(-1)
        batch_size = X.shape[0]
        batch_idx = torch.arange(0, batch_size)
        # batch_idx是[0,0,0,...,1,1,1,...]
        batch_idx = torch.repeat_interleave(batch_idx, num_pred_positions)
        # 提取被掩码位置的表示
        masked_X = X[batch_idx, pred_positions]
        masked_X = masked_X.reshape((batch_size, num_pred_positions, -1))
        mlm_Y_hat = self.mlp(masked_X)
        return mlm_Y_hat

# 测试MaskLM
mlm = MaskLM(vocab_size, num_hiddens)
mlm_positions = torch.tensor([[1, 5, 2], [6, 1, 5]])
mlm_Y_hat = mlm(encoded_X, mlm_positions)
print(f'MLM输出形状: {mlm_Y_hat.shape}')  # (batch_size, num_pred, vocab_size)

# 计算损失
mlm_Y = torch.tensor([[7, 8, 9], [10, 20, 30]])
loss = nn.CrossEntropyLoss(reduction='none')
mlm_l = loss(mlm_Y_hat.reshape((-1, vocab_size)), mlm_Y.reshape(-1))
print(f'MLM损失形状: {mlm_l.shape}')

### 2.2 下一句预测(NSP)

**动机**:
- 许多NLP任务涉及理解两个句子之间的关系
  - 问答: 问题 + 段落
  - 自然语言推理: 前提 + 假设
  - 文本蕴含: 句子对
- MLM只关注token级别,缺乏句子级别的理解

**任务定义**:
- 给定两个句子A和B
- 二分类: B是否是A的下一句?
- 标签:
  - `IsNext`: B确实跟在A后面(50%)
  - `NotNext`: B是随机采样的(50%)

**实现**:
- 使用`<cls>`token的表示作为整个句子对的表示
- 通过简单的MLP进行二分类

In [ ]:
# NextSentencePred实现
class NextSentencePred(nn.Module):
    """BERT的下一句预测任务"""
    def __init__(self, num_inputs, **kwargs):
        super(NextSentencePred, self).__init__(**kwargs)
        # 二分类:IsNext vs NotNext
        self.output = nn.Linear(num_inputs, 2)

    def forward(self, X):
        """
        参数:
            X: <cls> token的表示 (batch_size, num_hiddens)
        返回:
            nsp_Y_hat: 二分类logits (batch_size, 2)
        """
        return self.output(X)

# 测试NSP
# 使用<cls>位置(索引0)的编码
encoded_X_cls = torch.flatten(encoded_X, start_dim=1)
nsp = NextSentencePred(encoded_X_cls.shape[-1])
nsp_Y_hat = nsp(encoded_X_cls)
print(f'NSP输出形状: {nsp_Y_hat.shape}')  # (batch_size, 2)

# 计算损失
nsp_y = torch.tensor([0, 1])  # 第一个是IsNext, 第二个是NotNext
nsp_l = loss(nsp_Y_hat, nsp_y)
print(f'NSP损失形状: {nsp_l.shape}')

### 2.3 完整的BERT模型

**组合两个任务**:
- 总损失 = MLM损失 + NSP损失
- 联合训练两个任务
- 所有参数共享(除了任务特定的头)

In [ ]:
# 完整BERT模型
class BERTModel(nn.Module):
    """BERT模型"""
    def __init__(self, vocab_size, num_hiddens, norm_shape, ffn_num_input,
                 ffn_num_hiddens, num_heads, num_layers, dropout,
                 max_len=1000, key_size=768, query_size=768, value_size=768,
                 hid_in_features=768, mlm_in_features=768,
                 nsp_in_features=768):
        super(BERTModel, self).__init__()
        # BERT编码器
        self.encoder = BERTEncoder(vocab_size, num_hiddens, norm_shape,
                    ffn_num_input, ffn_num_hiddens, num_heads, num_layers,
                    dropout, max_len=max_len, key_size=key_size,
                    query_size=query_size, value_size=value_size)
        # NSP的隐藏层
        self.hidden = nn.Sequential(
            nn.Linear(hid_in_features, num_hiddens),
            nn.Tanh()
        )
        # MLM头
        self.mlm = MaskLM(vocab_size, num_hiddens, mlm_in_features)
        # NSP头
        self.nsp = NextSentencePred(nsp_in_features)

    def forward(self, tokens, segments, valid_lens=None,
                pred_positions=None):
        # 编码
        encoded_X = self.encoder(tokens, segments, valid_lens)
        # MLM预测
        if pred_positions is not None:
            mlm_Y_hat = self.mlm(encoded_X, pred_positions)
        else:
            mlm_Y_hat = None
        # NSP预测(使用<cls>的表示)
        nsp_Y_hat = self.nsp(self.hidden(encoded_X[:, 0, :]))
        return encoded_X, mlm_Y_hat, nsp_Y_hat

## 3. WikiText-2数据集准备

### 3.1 为什么选择WikiText-2?

**原始BERT使用的数据**:
- BooksCorpus: 8亿词
- 英文维基百科: 25亿词
- 总计: 33亿词, 需要大量计算资源

**WikiText-2优势**:
- 较小但高质量(来自维基百科)
- 保留原始标点、大小写、数字
- 适合演示和学习
- 比PTB大2倍以上

**WikiText-2 vs PTB**:
- WikiText-2: 保留标点,适合NSP任务
- PTB: 简化处理,不适合NSP

In [ ]:
# 注册WikiText-2数据集
d2l.DATA_HUB['wikitext-2'] = (
    'https://s3.amazonaws.com/research.metamind.io/wikitext/'
    'wikitext-2-v1.zip', '3c914d17d80b1459be871a5039ac23e752a53cbe')

def _read_wiki(data_dir):
    """读取WikiText-2数据集"""
    import os
    file_name = os.path.join(data_dir, 'wiki.train.tokens')
    with open(file_name, 'r') as f:
        lines = f.readlines()
    # 将大写转小写,用句号分句
    # 只保留至少有2句话的段落
    paragraphs = [line.strip().lower().split(' . ')
                  for line in lines if len(line.split(' . ')) >= 2]
    random.shuffle(paragraphs)
    return paragraphs

### 3.2 生成NSP数据

In [ ]:
def _get_next_sentence(sentence, next_sentence, paragraphs):
    """生成下一句预测任务的样本"""
    if random.random() < 0.5:
        is_next = True
    else:
        # 从随机段落中随机选择句子
        next_sentence = random.choice(random.choice(paragraphs))
        is_next = False
    return sentence, next_sentence, is_next

def _get_nsp_data_from_paragraph(paragraph, paragraphs, vocab, max_len):
    """从一个段落生成NSP训练样本"""
    nsp_data_from_paragraph = []
    for i in range(len(paragraph) - 1):
        tokens_a, tokens_b, is_next = _get_next_sentence(
            paragraph[i], paragraph[i + 1], paragraphs)
        # 考虑1个<cls>和2个<sep>
        if len(tokens_a) + len(tokens_b) + 3 > max_len:
            continue
        tokens, segments = get_tokens_and_segments(tokens_a, tokens_b)
        nsp_data_from_paragraph.append((tokens, segments, is_next))
    return nsp_data_from_paragraph

### 3.3 生成MLM数据

In [ ]:
def _replace_mlm_tokens(tokens, candidate_pred_positions, num_mlm_preds,
                        vocab):
    """替换token以生成MLM训练样本"""
    # 复制tokens
    mlm_input_tokens = [token for token in tokens]
    pred_positions_and_labels = []
    # 打乱候选位置
    random.shuffle(candidate_pred_positions)
    for mlm_pred_position in candidate_pred_positions:
        if len(pred_positions_and_labels) >= num_mlm_preds:
            break
        masked_token = None
        # 80%: 替换为<mask>
        if random.random() < 0.8:
            masked_token = '<mask>'
        else:
            # 10%: 保持不变
            if random.random() < 0.5:
                masked_token = tokens[mlm_pred_position]
            # 10%: 替换为随机token
            else:
                masked_token = random.choice(vocab.idx_to_token)
        mlm_input_tokens[mlm_pred_position] = masked_token
        pred_positions_and_labels.append(
            (mlm_pred_position, tokens[mlm_pred_position]))
    return mlm_input_tokens, pred_positions_and_labels

def _get_mlm_data_from_tokens(tokens, vocab):
    """从tokens生成MLM训练数据"""
    candidate_pred_positions = []
    # 排除特殊token
    for i, token in enumerate(tokens):
        if token in ['<cls>', '<sep>']:
            continue
        candidate_pred_positions.append(i)
    # 预测15%的token
    num_mlm_preds = max(1, round(len(tokens) * 0.15))
    mlm_input_tokens, pred_positions_and_labels = _replace_mlm_tokens(
        tokens, candidate_pred_positions, num_mlm_preds, vocab)
    pred_positions_and_labels = sorted(pred_positions_and_labels,
                                       key=lambda x: x[0])
    pred_positions = [v[0] for v in pred_positions_and_labels]
    mlm_pred_labels = [v[1] for v in pred_positions_and_labels]
    return vocab[mlm_input_tokens], pred_positions, vocab[mlm_pred_labels]

### 3.4 填充和批处理

In [ ]:
def _pad_bert_inputs(examples, max_len, vocab):
    """填充BERT输入序列"""
    max_num_mlm_preds = round(max_len * 0.15)
    all_token_ids, all_segments, valid_lens = [], [], []
    all_pred_positions, all_mlm_weights, all_mlm_labels = [], [], []
    nsp_labels = []
    
    for (token_ids, pred_positions, mlm_pred_label_ids, segments,
         is_next) in examples:
        # 填充token_ids
        all_token_ids.append(torch.tensor(
            token_ids + [vocab['<pad>']] * (max_len - len(token_ids)),
            dtype=torch.long))
        # 填充segments
        all_segments.append(torch.tensor(
            segments + [0] * (max_len - len(segments)),
            dtype=torch.long))
        # valid_lens不包括<pad>
        valid_lens.append(torch.tensor(len(token_ids), dtype=torch.float32))
        # 填充预测位置
        all_pred_positions.append(torch.tensor(
            pred_positions + [0] * (max_num_mlm_preds - len(pred_positions)),
            dtype=torch.long))
        # 填充权重(填充位置权重为0)
        all_mlm_weights.append(torch.tensor(
            [1.0] * len(mlm_pred_label_ids) + 
            [0.0] * (max_num_mlm_preds - len(pred_positions)),
            dtype=torch.float32))
        # 填充MLM标签
        all_mlm_labels.append(torch.tensor(
            mlm_pred_label_ids + 
            [0] * (max_num_mlm_preds - len(mlm_pred_label_ids)),
            dtype=torch.long))
        # NSP标签
        nsp_labels.append(torch.tensor(is_next, dtype=torch.long))
    
    return (all_token_ids, all_segments, valid_lens, all_pred_positions,
            all_mlm_weights, all_mlm_labels, nsp_labels)

### 3.5 WikiText数据集类

In [ ]:
class _WikiTextDataset(torch.utils.data.Dataset):
    """用于BERT预训练的WikiText-2数据集"""
    def __init__(self, paragraphs, max_len):
        # 分词
        paragraphs = [d2l.tokenize(paragraph, token='word')
                      for paragraph in paragraphs]
        sentences = [sentence for paragraph in paragraphs
                     for sentence in paragraph]
        # 构建词表(最小频率5,保留特殊token)
        self.vocab = d2l.Vocab(sentences, min_freq=5, reserved_tokens=[
            '<pad>', '<mask>', '<cls>', '<sep>'])
        
        # 生成NSP数据
        examples = []
        for paragraph in paragraphs:
            examples.extend(_get_nsp_data_from_paragraph(
                paragraph, paragraphs, self.vocab, max_len))
        
        # 生成MLM数据
        examples = [(_get_mlm_data_from_tokens(tokens, self.vocab) +
                     (segments, is_next))
                    for tokens, segments, is_next in examples]
        
        # 填充
        (self.all_token_ids, self.all_segments, self.valid_lens,
         self.all_pred_positions, self.all_mlm_weights,
         self.all_mlm_labels, self.nsp_labels) = _pad_bert_inputs(
            examples, max_len, self.vocab)

    def __getitem__(self, idx):
        return (self.all_token_ids[idx], self.all_segments[idx],
                self.valid_lens[idx], self.all_pred_positions[idx],
                self.all_mlm_weights[idx], self.all_mlm_labels[idx],
                self.nsp_labels[idx])

    def __len__(self):
        return len(self.all_token_ids)

def load_data_wiki(batch_size, max_len):
    """加载WikiText-2数据集"""
    num_workers = d2l.get_dataloader_workers()
    data_dir = d2l.download_extract('wikitext-2', 'wikitext-2')
    paragraphs = _read_wiki(data_dir)
    train_set = _WikiTextDataset(paragraphs, max_len)
    train_iter = torch.utils.data.DataLoader(
        train_set, batch_size, shuffle=True, num_workers=num_workers)
    return train_iter, train_set.vocab

In [ ]:
# 加载数据
batch_size, max_len = 512, 64
train_iter, vocab = load_data_wiki(batch_size, max_len)

# 查看一个批次
for (tokens_X, segments_X, valid_lens_x, pred_positions_X, mlm_weights_X,
     mlm_Y, nsp_y) in train_iter:
    print(f'tokens_X形状: {tokens_X.shape}')
    print(f'segments_X形状: {segments_X.shape}')
    print(f'valid_lens_x形状: {valid_lens_x.shape}')
    print(f'pred_positions_X形状: {pred_positions_X.shape}')
    print(f'mlm_weights_X形状: {mlm_weights_X.shape}')
    print(f'mlm_Y形状: {mlm_Y.shape}')
    print(f'nsp_y形状: {nsp_y.shape}')
    print(f'\n词表大小: {len(vocab)}')
    break

## 4. BERT预训练

### 4.1 模型配置

**原始BERT配置**:
- BERT-Base: 12层, 768隐藏, 12头, 110M参数
- BERT-Large: 24层, 1024隐藏, 16头, 340M参数

**我们的演示配置** (计算资源有限):
- 2层, 128隐藏, 2头
- 足够演示原理

In [ ]:
# 创建BERT模型
net = BERTModel(
    len(vocab), num_hiddens=128, norm_shape=[128],
    ffn_num_input=128, ffn_num_hiddens=256, num_heads=2,
    num_layers=2, dropout=0.2, key_size=128, query_size=128,
    value_size=128, hid_in_features=128, mlm_in_features=128,
    nsp_in_features=128)

devices = d2l.try_all_gpus()
loss = nn.CrossEntropyLoss()

### 4.2 训练函数

In [ ]:
def _get_batch_loss_bert(net, loss, vocab_size, tokens_X,
                         segments_X, valid_lens_x,
                         pred_positions_X, mlm_weights_X,
                         mlm_Y, nsp_y):
    """计算一个批次的BERT损失"""
    # 前向传播
    _, mlm_Y_hat, nsp_Y_hat = net(
        tokens_X, segments_X, valid_lens_x.reshape(-1), pred_positions_X)
    
    # MLM损失
    mlm_l = loss(mlm_Y_hat.reshape(-1, vocab_size), mlm_Y.reshape(-1)) * \
            mlm_weights_X.reshape(-1, 1)
    mlm_l = mlm_l.sum() / (mlm_weights_X.sum() + 1e-8)
    
    # NSP损失
    nsp_l = loss(nsp_Y_hat, nsp_y)
    
    # 总损失
    l = mlm_l + nsp_l
    return mlm_l, nsp_l, l

def train_bert(train_iter, net, loss, vocab_size, devices, num_steps):
    """预训练BERT"""
    net = nn.DataParallel(net, device_ids=devices).to(devices[0])
    trainer = torch.optim.Adam(net.parameters(), lr=0.01)
    step, timer = 0, d2l.Timer()
    animator = d2l.Animator(
        xlabel='step', ylabel='loss',
        xlim=[1, num_steps], legend=['mlm', 'nsp'])
    # 累加器: MLM损失, NSP损失, 样本数, 批次数
    metric = d2l.Accumulator(4)
    num_steps_reached = False
    
    while step < num_steps and not num_steps_reached:
        for tokens_X, segments_X, valid_lens_x, pred_positions_X, \
            mlm_weights_X, mlm_Y, nsp_y in train_iter:
            # 移动到设备
            tokens_X = tokens_X.to(devices[0])
            segments_X = segments_X.to(devices[0])
            valid_lens_x = valid_lens_x.to(devices[0])
            pred_positions_X = pred_positions_X.to(devices[0])
            mlm_weights_X = mlm_weights_X.to(devices[0])
            mlm_Y, nsp_y = mlm_Y.to(devices[0]), nsp_y.to(devices[0])
            
            trainer.zero_grad()
            timer.start()
            mlm_l, nsp_l, l = _get_batch_loss_bert(
                net, loss, vocab_size, tokens_X, segments_X, valid_lens_x,
                pred_positions_X, mlm_weights_X, mlm_Y, nsp_y)
            l.backward()
            trainer.step()
            metric.add(mlm_l, nsp_l, tokens_X.shape[0], 1)
            timer.stop()
            animator.add(step + 1,
                         (metric[0] / metric[3], metric[1] / metric[3]))
            step += 1
            if step == num_steps:
                num_steps_reached = True
                break

    print(f'MLM loss {metric[0] / metric[3]:.3f}, '
          f'NSP loss {metric[1] / metric[3]:.3f}')
    print(f'{metric[2] / timer.sum():.1f} sentence pairs/sec on '
          f'{str(devices)}')

In [ ]:
# 开始预训练(演示用,只训练50步)
train_bert(train_iter, net, loss, len(vocab), devices, 50)

## 5. 使用预训练BERT

### 5.1 提取BERT表示

In [ ]:
def get_bert_encoding(net, tokens_a, tokens_b=None):
    """获取BERT编码表示"""
    tokens, segments = get_tokens_and_segments(tokens_a, tokens_b)
    token_ids = torch.tensor(
        vocab[tokens], device=devices[0]).unsqueeze(0)
    segments = torch.tensor(
        segments, device=devices[0]).unsqueeze(0)
    valid_len = torch.tensor(
        len(tokens), device=devices[0]).unsqueeze(0)
    encoded_X, _, _ = net(token_ids, segments, valid_len)
    return encoded_X

### 5.2 单句表示

In [ ]:
# 单句: "a crane is flying"
tokens_a = ['a', 'crane', 'is', 'flying']
encoded_text = get_bert_encoding(net, tokens_a)

# 提取<cls>和"crane"的表示
encoded_text_cls = encoded_text[:, 0, :]  # <cls>在位置0
encoded_text_crane = encoded_text[:, 2, :]  # crane在位置2

print(f'编码形状: {encoded_text.shape}')
print(f'<cls>表示形状: {encoded_text_cls.shape}')
print(f'"crane"表示的前3个元素: {encoded_text_crane[0][:3]}')

### 5.3 句对表示与上下文敏感性

In [ ]:
# 句对: "a crane driver came" + "he just left"
tokens_a = ['a', 'crane', 'driver', 'came']
tokens_b = ['he', 'just', 'left']
encoded_pair = get_bert_encoding(net, tokens_a, tokens_b)

# 提取表示
encoded_pair_cls = encoded_pair[:, 0, :]
encoded_pair_crane = encoded_pair[:, 2, :]  # crane在位置2

print(f'句对编码形状: {encoded_pair.shape}')
print(f'<cls>表示形状: {encoded_pair_cls.shape}')
print(f'"crane"表示的前3个元素: {encoded_pair_crane[0][:3]}')

# 比较两个上下文中"crane"的表示
print(f'\n上下文敏感性检验:')
print(f'单句中"crane": {encoded_text_crane[0][:3]}')
print(f'句对中"crane": {encoded_pair_crane[0][:3]}')
print(f'\n注意:两个表示不同,说明BERT是上下文敏感的!')

## 6. 小结

### 6.1 BERT的核心创新

**1. 双向编码**
- 通过掩码语言模型实现真正的双向上下文
- 优于GPT的单向编码
- 优于ELMo的浅层双向拼接

**2. 预训练任务**
- MLM: token级别的语义理解
- NSP: 句子级别的关系建模
- 联合训练,相互促进

**3. 任务无关架构**
- 最小化下游任务的架构改动
- 只需添加简单的输出层
- 端到端微调所有参数

**4. 上下文敏感表示**
- 同一词在不同上下文有不同表示
- 解决一词多义问题
- 捕捉细粒度语义

### 6.2 BERT模型规格

| 模型 | 层数 | 隐藏单元 | 注意力头 | 参数量 |
|------|------|----------|----------|--------|
| BERT-Base | 12 | 768 | 12 | 110M |
| BERT-Large | 24 | 1024 | 16 | 340M |

### 6.3 预训练数据

- BooksCorpus: 8亿词
- 英文维基百科: 25亿词
- 总计: 33亿词
- 预训练时间: 4天(16个TPU) 或 数周(GPU)

### 6.4 下游应用

**BERT在11个NLP任务上刷新SOTA**:
1. 单句分类: SST-2情感分析
2. 句对分类: MNLI推理, QQP问题匹配
3. 问答: SQuAD
4. 序列标注: NER命名实体识别

**微调策略**:
- 单句/句对: 使用`<cls>`表示 + 分类层
- 问答: 预测答案span的起始和结束位置
- 序列标注: 每个token的表示 + 标注层

### 6.5 BERT的影响

**后续发展**:
- RoBERTa: 改进的预训练策略
- ALBERT: 参数共享,减少模型大小
- ELECTRA: 判别式预训练
- GPT-3: 扩大模型规模到175B参数

**应用领域**:
- 搜索引擎: Google搜索使用BERT
- 问答系统: 客服机器人
- 文本生成: 摘要、翻译
- 多模态: ViLBERT(视觉+语言)

### 6.6 关键要点

1. **预训练-微调范式**
   - 大规模无监督预训练
   - 小规模有监督微调
   - 显著提升少样本学习能力

2. **输入表示**
   - Token + Segment + Position嵌入
   - 灵活支持单句和句对

3. **特殊token的作用**
   - `<cls>`: 句子/句对的聚合表示
   - `<sep>`: 分隔不同序列
   - `<mask>`: MLM任务的掩码

4. **MLM的掩码策略**
   - 80% `<mask>`: 主要学习目标
   - 10% 随机: 增加鲁棒性
   - 10% 不变: 减少预训练-微调差异

### 练习

1. 为什么MLM损失通常高于NSP损失?
   - 提示: 考虑任务难度和词表大小

2. 尝试将最大序列长度设为512(原始BERT配置)
   - 观察内存使用和训练速度

3. 实现BERT在情感分类任务上的微调
   - 使用`<cls>`表示 + 线性分类层

4. 比较BERT和GPT的区别
   - 编码器vs解码器
   - 双向vs单向
   - MLM vs 自回归语言模型

5. 研究BERT的变体(RoBERTa, ALBERT, ELECTRA)
   - 它们改进了什么?
   - 效果如何?